# GOT-OCR 2.0 — Kaggle API Server

## Kullanım
1. **Notebook Settings** -> **Accelerator** -> **GPU T4 x2** seçin
2. **Settings** -> **Internet** -> **On** (açık olduğuna emin olun)
3. Tüm hücreleri sırayla çalıştırın (Run All)
4. Son hücredeki **trycloudflare.com** URL'ini kopyalayın
5. Streamlit uygulamasındaki **GOT-OCR API URL** kutusuna yapıştırın
6. OCR Motoru olarak **GOT-OCR API**'yi seçin

## Önemli Uyarılar
- Kaggle GPU kotası: ~30 saat/hafta, oturum başına ~9 saat
- URL her yeniden başlatmada değişir, Streamlit'ten güncellemeniz gerekir
- Notebook'u açık tutun, aksi halde bağlantı kopar

In [ ]:
!pip install -q transformers verovio fastapi uvicorn python-multipart Pillow nest_asyncio requests
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os, sys
from transformers import AutoProcessor
import torch

MODEL_ID = 'stepfun-ai/GOT-OCR2_0'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32

# Model kod dosyalarini klonla (agir .bin dosyalari HARIC)
!GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://huggingface.co/stepfun-ai/GOT-OCR2_0 /content/GOT-OCR2_0 2>/dev/null
sys.path.insert(0, '/content/GOT-OCR2_0')

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

from modeling_GOT import ForCausalLM

print(f'Model yukleniyor... (device={DEVICE}, dtype={DTYPE})')
model = ForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True, device_map='auto'
).to(DEVICE)
print('Model yuklendi!')

In [ ]:
from PIL import Image
import io

def run_ocr(image_bytes, max_new_tokens=4096):
    image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    prompt = 'OCR with format: plain text'
    inputs = processor(image, prompt, return_tensors='pt').to(DEVICE, dtype=DTYPE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, eos_token_id=processor.tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True)

print('OCR fonksiyonu hazir!')

In [ ]:
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import uvicorn, nest_asyncio

nest_asyncio.apply()
app = FastAPI(title='GOT-OCR API')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

@app.get('/health')
async def health():
    return {'status': 'ok', 'model': 'GOT-OCR2_0', 'device': DEVICE}

@app.post('/ocr')
async def ocr_endpoint(file: UploadFile = File(...)):
    try:
        contents = await file.read()
        if len(contents) > 10 * 1024 * 1024:
            raise HTTPException(status_code=400, detail='Dosya cok buyuk (max 10MB)')
        return {'text': run_ocr(contents), 'status': 'ok'}
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# Sunucuyu arka planda baslat
import threading, time
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print('Sunucu baslatildi! Bir sonraki hucrede URL olusturulacak.')

In [ ]:
# Cloudflare tunnel — ucretsiz, kayit gerekmez
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import subprocess, time, re

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

# URL'yi bekle
public_url = None
for _ in range(30):
    line = proc.stdout.readline()
    if not line:
        time.sleep(1)
        continue
    print(line.strip())
    match = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
    if match:
        public_url = match.group(1)
        break

if public_url:
    print(f'\n{"="*60}')
    print(f'GOT-OCR API AKTIF!')
    print(f'Public URL: {public_url}')
    print(f'\nBu URLyi Streamlit appindeki GOT-OCR API URL alanina yapistirin.')
    print(f'OCR Motoru olarak "GOT-OCR API" secmeyi unutmayin.')
    print(f'{"="*60}')
else:
    print('URL alinamadi! Cloudflared loglarini kontrol edin.')
    print('Internet baglantisi acik mi kontrol edin: Settings > Internet > On')

In [ ]:
import requests
try:
    r = requests.get(f'{public_url}/health', timeout=5)
    print(f'Health: {r.json()}')
except Exception as e:
    print(f'Hata: {e}, 5sn bekleyip tekrar deneyin.')